# Detecção de Jogadores de Padel com YOLOv8

Este notebook implementa um sistema de detecção de jogadores de padel usando YOLOv8 e segue as etapas:
1. Preparação do ambiente
2. Coleta e extração de frames de vídeos
3. Anotação de dados (usando Roboflow)
4. Treinamento do modelo YOLOv8
5. Teste e avaliação do modelo

## Etapa 1: Preparação do Ambiente

Primeiro, vamos verificar se temos acesso a uma GPU e instalar as bibliotecas necessárias.

In [ ]:
# Verificar se temos acesso a GPU
!nvidia-smi

In [ ]:
# Instalar bibliotecas necessárias
!pip install ultralytics pytube opencv-python matplotlib

In [ ]:
# Importar bibliotecas
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pytube import YouTube
from google.colab import drive
from IPython.display import display, Image

## Etapa 2: Coleta e Extração de Frames de Vídeos

Vamos baixar alguns vídeos de partidas de padel do YouTube e extrair frames para criar nosso dataset.

In [ ]:
# Montar o Google Drive para salvar os dados
drive.mount('/content/drive')

# Criar diretórios para armazenar os dados
base_dir = '/content/drive/MyDrive/padel_detection'
videos_dir = f'{base_dir}/videos'
frames_dir = f'{base_dir}/frames'

os.makedirs(videos_dir, exist_ok=True)
os.makedirs(frames_dir, exist_ok=True)

In [ ]:
# Lista de URLs de vídeos de padel no YouTube
video_urls = [
    'https://www.youtube.com/watch?v=example1',  # Substitua pelos URLs reais
    'https://www.youtube.com/watch?v=example2',
    'https://www.youtube.com/watch?v=example3',
    'https://www.youtube.com/watch?v=example4',
    'https://www.youtube.com/watch?v=example5'
]

# Função para baixar vídeos do YouTube
# Modified download function
def download_youtube_video(url, output_path):
    try:
        # Add a retry mechanism
        for attempt in range(3):
            try:
                yt = YouTube(url)
                # Print more debug info
                print(f"Attempting to download: {url}")
                print(f"Available streams: {len(yt.streams.filter())}")
                
                video = yt.streams.filter(progressive=True, file_extension='mp4')
                if not video:
                    print("No progressive MP4 streams found, trying any MP4")
                    video = yt.streams.filter(file_extension='mp4')
                
                if not video:
                    print("No MP4 streams found, trying any format")
                    video = yt.streams
                
                video = video.order_by('resolution').desc().first()
                
                if not video:
                    print("No streams available")
                    return None
                    
                print(f"Baixando: {yt.title} ({video.resolution})")
                video.download(output_path=output_path)
                filename = video.default_filename
                return os.path.join(output_path, filename)
            except Exception as e:
                print(f"Attempt {attempt+1} failed: {e}")
                if attempt < 2:  # If not the last attempt
                    print("Retrying...")
                else:
                    raise  # Re-raise on last attempt
    except Exception as e:
        print(f"Erro ao baixar vídeo {url}: {e}")
        return None

In [ ]:
# Baixar os vídeos
video_paths = []
for url in video_urls:
    video_path = download_youtube_video(url, videos_dir)
    if video_path:
        video_paths.append(video_path)

print(f"Total de vídeos baixados: {len(video_paths)}")

In [ ]:
# Função para extrair frames de um vídeo
def extract_frames(video_path, output_dir, frame_interval=30):
    """
    Extrai frames de um vídeo a cada frame_interval frames.
    """
    video_name = os.path.basename(video_path).split('.')[0]
    video_frames_dir = os.path.join(output_dir, video_name)
    os.makedirs(video_frames_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_count % frame_interval == 0:
            frame_path = os.path.join(video_frames_dir, f"frame_{saved_count:04d}.jpg")
            cv2.imwrite(frame_path, frame)
            saved_count += 1
            
        frame_count += 1
        
    cap.release()
    print(f"Extraídos {saved_count} frames do vídeo {video_name}")
    return saved_count

In [ ]:
# Extrair frames de cada vídeo
total_frames = 0
for video_path in video_paths:
    frames = extract_frames(video_path, frames_dir, frame_interval=60)  # Extrair um frame a cada 2 segundos (considerando 30fps)
    total_frames += frames

print(f"Total de frames extraídos: {total_frames}")

## Etapa 3: Anotação de Dados

Nesta etapa, você deve usar o Roboflow para anotar os frames extraídos. Siga estes passos:

1. Crie uma conta no [Roboflow](https://roboflow.com/)
2. Crie um novo projeto e selecione "Object Detection" como tipo
3. Faça upload dos frames extraídos
4. Anote os jogadores de padel em cada frame
5. Gere o dataset e exporte-o no formato YOLOv8

Depois de anotar os dados, você receberá um código para baixar o dataset. Substitua a variável `ROBOFLOW_KEY` abaixo pelo seu código.

In [ ]:
# Baixar o dataset anotado do Roboflow
# Substitua a linha abaixo pelo código fornecido pelo Roboflow após a anotação
ROBOFLOW_KEY = "sua_chave_do_roboflow_aqui"

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="GHr9ANaYSrz0jyDxIIBp")
project = rf.workspace("objectdetection-bqixx").project("padel-player-detection-fofih")
version = project.version(1)
dataset = version.download("yolov8")
                

## Etapa 4: Treinamento do Modelo YOLOv8

Agora vamos treinar o modelo YOLOv8 com os dados anotados.

In [ ]:
# Definir o diretório do dataset
dataset_dir = dataset.location  # Obtido do download do Roboflow

# Criar arquivo de configuração YAML para o treinamento
yaml_content = f"""
path: {dataset_dir}
train: train/images
val: valid/images
test: test/images

names:
  0: player
"""

yaml_path = os.path.join(dataset_dir, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"Arquivo de configuração criado em {yaml_path}")

In [ ]:
# Baixar o modelo pré-treinado YOLOv8n
model = YOLO('yolov8n.pt')

In [ ]:
# Treinar o modelo
!yolo task=detect mode=train model=yolov8n.pt data={yaml_path} epochs=50 imgsz=640 batch=16 patience=10

## Etapa 5: Teste e Avaliação do Modelo

Vamos testar o modelo treinado em algumas imagens e vídeos.

In [ ]:
# Carregar o modelo treinado
model_path = "runs/detect/train/weights/best.pt"
model = YOLO(model_path)

# Verificar se o modelo foi carregado corretamente
print(f"Modelo carregado: {model_path}")

In [ ]:
# Testar o modelo em algumas imagens do conjunto de validação
val_images_dir = os.path.join(dataset_dir, 'valid/images')
test_images = os.listdir(val_images_dir)[:5]  # Testar nas primeiras 5 imagens

plt.figure(figsize=(15, 10))
for i, img_name in enumerate(test_images):
    img_path = os.path.join(val_images_dir, img_name)
    results = model(img_path)
    
    # Plotar a imagem com as detecções
    plt.subplot(2, 3, i+1)
    plt.imshow(cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB))
    plt.title(f"Imagem {i+1}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Testar o modelo em um vídeo
def process_video(video_path, output_path):
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # Definir o codec e criar objeto VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Processar apenas 1 frame a cada 3 para acelerar
        if frame_count % 3 == 0:
            results = model(frame)
            frame_with_boxes = results[0].plot()
            out.write(frame_with_boxes)
        else:
            out.write(frame)
            
        frame_count += 1
        if frame_count % 100 == 0:
            print(f"Processados {frame_count} frames")
    
    cap.release()
    out.release()
    print(f"Vídeo processado e salvo em {output_path}")

In [ ]:
# Processar um vídeo de teste (você pode usar um dos vídeos baixados anteriormente)
if len(video_paths) > 0:
    test_video = video_paths[0]
    output_video = os.path.join(base_dir, 'resultado_deteccao.mp4')
    process_video(test_video, output_video)

## Avaliação do Modelo

Vamos avaliar o desempenho do modelo no conjunto de validação.

In [ ]:
# Avaliar o modelo
!yolo task=detect mode=val model={model_path} data={yaml_path}

## Salvar o Modelo Treinado

Vamos salvar o modelo treinado no Google Drive para uso futuro.

In [ ]:
# Copiar o modelo para o Google Drive
model_save_path = f"{base_dir}/padel_player_detector.pt"
!cp {model_path} {model_save_path}
print(f"Modelo salvo em: {model_save_path}")

## Conclusão

Neste notebook, implementamos um sistema completo para detecção de jogadores de padel usando YOLOv8:

1. Preparamos o ambiente no Google Colab
2. Baixamos vídeos de partidas de padel e extraímos frames
3. Anotamos os dados usando Roboflow
4. Treinamos um modelo YOLOv8 personalizado
5. Testamos e avaliamos o modelo em imagens e vídeos

O modelo treinado pode ser usado para detectar jogadores em vídeos de partidas de padel, o que pode ser útil para análise de jogos, estatísticas e treinamento.